# AutoArchitect Brain — Core 1: Task Understander
## Fine-tuning Qwen2.5-1.5B-Instruct with QLoRA

**Day 18 of AutoArchitect 28-Day Sprint**

This notebook fine-tunes a 1.5B parameter model to understand ML problem descriptions
and output structured analysis JSON (domain, complexity, intent, etc.).

**Requirements:** T4 GPU runtime (Runtime → Change runtime type → T4 GPU)  
**Time:** ~30-45 minutes  
**Dataset:** 672 examples from DeepSeek V3 teacher

## Step 1: Install Dependencies

In [ ]:
# Install Unsloth (fast QLoRA) and training stack
# This takes ~3 minutes on first run
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q bitsandbytes peft accelerate
!pip install -q transformers datasets trl
print('Installation complete.')

In [ ]:
# Verify GPU is available
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NOT AVAILABLE"}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
assert torch.cuda.is_available(), 'ERROR: No GPU found. Change runtime to T4 GPU.'

## Step 2: Upload Dataset

In [ ]:
# Upload dataset_task_understander.jsonl from your local machine
from google.colab import files
print('Select dataset_task_understander.jsonl from your computer...')
uploaded = files.upload()

# Verify upload
import json
dataset_path = 'dataset_task_understander.jsonl'
with open(dataset_path) as f:
    examples = [json.loads(l) for l in f if l.strip()]
print(f'Loaded {len(examples)} training examples')
print('Sample input:', examples[0]['input'][:80])
print('Sample output:', examples[0]['output'])

## Step 3: Load Base Model with QLoRA (4-bit)

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
BASE_MODEL = 'Qwen/Qwen2.5-1.5B-Instruct'

print(f'Loading {BASE_MODEL} with 4-bit quantization...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,          # auto-detect
    load_in_4bit=True,   # QLoRA
)
print(f'Model loaded. Parameters: {sum(p.numel() for p in model.parameters()):,}')

## Step 4: Apply LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')

## Step 5: Load and Format Dataset

In [ ]:
import json
from datasets import load_dataset

SYSTEM_PROMPT = (
    "You are AutoArchitect's Task Understander. "
    "Given an ML problem description, output a structured JSON analysis "
    "with fields: primary_intent, domain, complexity, real_time_required, "
    "multi_modal, and key_entities."
)

def format_example(ex):
    """Format as Qwen2.5-Instruct chat template."""
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": f"Problem: {ex['input']}"},
        {"role": "assistant", "content": json.dumps(ex['output'])},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

raw_ds = load_dataset("json", data_files=dataset_path)["train"]
ds     = raw_ds.map(format_example, remove_columns=raw_ds.column_names)

# 90/10 train/eval split
split    = ds.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
eval_ds  = split["test"]

print(f'Train: {len(train_ds)}  Eval: {len(eval_ds)}')
print('\nFormatted example (first 300 chars):')
print(train_ds[0]["text"][:300])

## Step 6: Configure Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

OUTPUT_DIR = "core1_output"

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_steps=100,
        save_total_limit=2,
        output_dir=OUTPUT_DIR,
        fp16=False,
        bf16=True,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
    ),
)

print('Trainer configured.')
print(f'  Effective batch size: {4 * 4} examples/step')
print(f'  Steps per epoch: {len(train_ds) // (4 * 4)}')
print(f'  Total steps: {3 * len(train_ds) // (4 * 4)}')

## Step 7: Train

In [ ]:
import time

print('Starting training...')
print('Expected time: 30-45 minutes on T4 GPU')
print('=' * 50)

t0 = time.time()
trainer_stats = trainer.train()
duration = round(time.time() - t0, 1)

print('=' * 50)
print(f'Training complete in {duration:.0f}s ({duration/60:.1f} min)')
print(f'Final train loss: {trainer_stats.training_loss:.4f}')

## Step 8: Save LoRA Adapter

In [ ]:
ADAPTER_DIR = "core1_task_understander_adapter"

model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

import os
size_mb = sum(
    os.path.getsize(os.path.join(r, f))
    for r, _, files in os.walk(ADAPTER_DIR)
    for f in files
) / (1024 * 1024)

print(f'Adapter saved to: {ADAPTER_DIR}')
print(f'Adapter size: {size_mb:.1f} MB')
print('Files:', os.listdir(ADAPTER_DIR))

## Step 9: Evaluate on 10 Unseen Problems

In [ ]:
import json
from unsloth import FastLanguageModel

# Switch to inference mode
FastLanguageModel.for_inference(model)

REQUIRED_FIELDS = [
    "primary_intent", "domain", "complexity",
    "real_time_required", "multi_modal", "key_entities"
]

def predict(problem_text):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"Problem: {problem_text}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    decoded = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                               skip_special_tokens=True)
    try:
        # Extract JSON from response
        start = decoded.find('{')
        end   = decoded.rfind('}') + 1
        return json.loads(decoded[start:end]) if start >= 0 else None
    except Exception:
        return None

# Use eval set for testing (unseen during training)
test_examples = [json.loads(l) for l in open(dataset_path) if l.strip()]
# Take last 10 as unseen test (eval split used test_size=0.1 from end with seed)
import random
random.seed(99)
test_cases = random.sample(test_examples, 10)

print('Testing on 10 unseen problems...')
print('=' * 60)

schema_pass = 0
domain_match = 0

for i, ex in enumerate(test_cases):
    pred = predict(ex['input'])
    expected = ex['output']

    # Schema validation
    valid = (pred is not None and
             all(f in pred for f in REQUIRED_FIELDS))
    schema_pass += int(valid)

    # Domain match
    domain_ok = (pred is not None and
                 pred.get('domain') == expected.get('domain'))
    domain_match += int(domain_ok)

    status = 'OK' if valid else 'FAIL'
    print(f'[{i+1:02d}] [{status}] Input: {ex["input"][:50]}...')
    if pred:
        print(f'      Predicted domain: {pred.get("domain")}  '
              f'Expected: {expected.get("domain")}  '
              f'Match: {domain_ok}')
    else:
        print(f'      ERROR: Could not parse JSON response')

print('=' * 60)
print(f'Schema validation pass rate: {schema_pass}/10 ({schema_pass*10}%)')
print(f'Domain accuracy:             {domain_match}/10 ({domain_match*10}%)')

## Step 10: Download Adapter

In [ ]:
# Zip and download the adapter
!zip -r core1_adapter.zip core1_task_understander_adapter/

import os
zip_size_mb = os.path.getsize('core1_adapter.zip') / (1024 * 1024)
print(f'Adapter zip: {zip_size_mb:.1f} MB')

from google.colab import files
print('Downloading core1_adapter.zip...')
files.download('core1_adapter.zip')
print('Download started. Place the extracted folder in:')
print('  models/brain_cores/core1_task_understander_adapter/')

## Training Summary

After training completes, record these results and share with Claude Code:

| Metric | Value |
|--------|-------|
| Final train loss | _(fill in)_ |
| Schema pass rate | _(fill in)_ |
| Domain accuracy | _(fill in)_ |
| Training time | _(fill in)_ min |
| Adapter size | _(fill in)_ MB |

Next step: Tell Claude Code **"Core 1 is ready"** and place `core1_adapter.zip` in `models/brain_cores/`.